# EXT 셀 ROI 크롭 — 학습 전용

원본 1920×1080에서 **셀 영역만 잘라** 학습한다(유효 해상도 ×1.98, VRAM 부담 0).

| 셀 | 내용 | 필요 |
|---|---|---|
| §0 | 셋업 (경로·manifest·가중치) | — |
| §1 | **크롭 함수** (학습·추론 공용) | — |
| §2 | 데이터셋 빌드 (zip 스트리밍, 63GB 안 풂) | 디스크 ~10GB |
| §2.5 | 빌드 검증 + Drive 백업 | ~3분 |
| §3 | **재학습** | A100/L4 |

**데이터셋이 이미 `/content/work/ext_crop`에 있으면 §3만 실행하면 된다.**
평가는 전부 `battery_ext_crop_eval.ipynb`에서 한다.

## 현재 설정 = 실험 E (1클래스 검출기)
`RUN=train_ext_crop_v5_1cls` · `SINGLE_CLASS=True` · 8에폭 · **증강은 v2 그대로**
(v3에서 mosaic·회전을 껐다가 recall이 반토막 났다 — 0802 §8.1)

⚠️ **실행 전 `eval §18`로 E의 천장을 먼저 확인할 것.** 챔피언 v2보다 낮으면 이 학습은 낭비다.

## 설계 원칙
- **크롭 함수는 하나뿐** — 학습과 추론이 같은 함수를 호출한다(CT의 학습/추론 불일치 재발 차단)
- **GT 아웃라인을 쓰지 않는다** — 채도 기반 CV라 추론 시에도 라벨 없이 재현된다
- 1클래스 빌드는 **원본 라벨을 덮어쓰지 않는다**(별도 `_1cls` 디렉터리)


In [ ]:
# == §0 셋업 ==
!pip -q install ultralytics pandas
import os, io as _io, re, json as _j, zipfile, time, random, shutil
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
from google.colab import drive
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
Image.MAX_IMAGE_PIXELS = None

# ── Drive 탐색 (공유문서함·바로가기 포함) ──────────────────────────────────
DRIVE = Path('/content/drive')
DRIVE_ROOTS = [DRIVE/'MyDrive', DRIVE/'MyDrive/battery_yolo', DRIVE/'MyDrive/battery_yolo/data',
               DRIVE/'Shareddrives', DRIVE/'공유 드라이브', DRIVE/'.shortcut-targets-by-id']
DRIVE_ROOTS = [r for r in DRIVE_ROOTS if r.exists()]
_FIND_CACHE = {}
def find_any(pat, maxdepth=5):
    """Drive FUSE 재귀 글롭은 비싸다 — 같은 패턴은 캐시한다."""
    if pat in _FIND_CACHE: return _FIND_CACHE[pat]
    seen, out = set(), []
    for r in DRIVE_ROOTS:
        for d in range(maxdepth+1):
            try: cand = sorted(r.glob('*/'*d + pat))
            except Exception: cand = []
            for p in cand:
                if str(p) not in seen: seen.add(str(p)); out.append(p)
    _FIND_CACHE[pat] = out
    return out

# ★★ 버전 고정 — v4.1 zip과 manifest가 같은 드라이브에 그대로 있다.
#    안 박으면 "EXT 행이 있는 첫 manifest"로 v4.1을 잡고 조용히 구 데이터로 학습한다.
DATA_VER = 'v4_2'                     # 경로·파일명에 이 문자열이 있어야 채택
_VTAG = [DATA_VER, DATA_VER.replace('_', ''), DATA_VER.replace('_', '.')]   # v4_2 / v42 / v4.2
def _is_ver(p): return any(t in str(p).lower() for t in _VTAG)

def dedup(ps):
    """MyDrive 경로와 .shortcut-targets-by-id 경로는 같은 실물이다.
       94GB zip을 두 번 인덱싱하지 않도록 (파일명, 크기)로 접는다."""
    seen, out = set(), []
    for p in ps:
        try: k = (p.name, p.stat().st_size)
        except OSError: continue
        if k in seen: continue
        seen.add(k); out.append(p)
    return out

_cands = [m for m in find_any('manifest.csv') if _is_ver(m)]
assert _cands, (f'★{DATA_VER} manifest 못찾음. 발견된 manifest: '
                + str([str(m) for m in find_any('manifest.csv')]))
MANI = dedup(_cands)[0]
print(f'manifest [{DATA_VER}]:', MANI)
if len(dedup(_cands)) > 1:
    print('  ⚠️ 같은 버전 manifest가 여러 개 —', [str(m) for m in dedup(_cands)])
# ★필요 컬럼만 — manifest는 268MB·44컬럼이라 전량 읽으면 문자열로 ~2GB를 먹고,
#   §2에서 8워커를 fork할 때 copy-on-write가 깨져 OOM으로 세션이 죽을 수 있다.
USECOLS = ['modality', 'battery_id', 'application', 'split_role',
           'output_image_name', 'output_label_stem',
           'has_damaged', 'has_pollution', 'included_det', 'included_seg']
mani = pd.read_csv(MANI, dtype=str, keep_default_na=False, usecols=USECOLS)
EXT = mani[mani['modality'] == 'EXT'].copy()
del mani
print(f'  manifest EXT {len(EXT):,}행 · {len(USECOLS)}컬럼만 적재')
# ★v4.2: included_det / included_seg 가 다르다. seg 제외분 8,775장 중 50.9%가 Damaged
#   (크랙은 가늘고 길어 폴리곤 복구가 잘 깨짐) → det로 학습하면 Damaged 이미지를 11% 더 쓴다.
for t in ('det', 'seg'):
    s = EXT[EXT[f'included_{t}'].str.lower() == 'true']
    d = (s['has_damaged'].str.lower() == 'true')
    print(f'  included_{t}: ' + ' '.join(
        f'{k} {v:,}' for k, v in s['split_role'].value_counts().items())
        + f'  | Damaged보유 {d.sum():,}')
# (구 `seg` 변수는 제거 — §2가 TASK에 따라 SEL을 직접 만든다)

# ── zip 목록 (스트리밍으로 읽을 것) ────────────────────────────────────────
ZIPS = dedup([z for z in find_any('*.zip')
              if ('EXT' in z.name.upper() or 'RGB' in z.name.upper()) and _is_ver(z)])
assert ZIPS, f'★{DATA_VER} EXT zip 없음 — DATA_VER 확인'
for z in ZIPS: print(f'  {z.name}  {z.stat().st_size/1e9:.1f} GB')
_other = [z.name for z in find_any('*.zip')
          if ('EXT' in z.name.upper() or 'RGB' in z.name.upper()) and not _is_ver(z)]
if _other: print(f'  (제외된 타버전 zip: {sorted(set(_other))})')

# ★로컬 사본이 있으면 그걸 쓴다 (Drive FUSE 무작위 읽기가 느릴 때의 탈출구).
_local = []
for z in ZIPS:
    lp = Path('/content')/z.name
    if lp.exists() and lp.stat().st_size == z.stat().st_size: _local.append(lp)
    else: _local.append(z)
if any(p.parent == Path('/content') for p in _local):
    ZIPS = _local
    print('  ▶ 로컬 사본 사용:', [str(p) for p in ZIPS if p.parent == Path('/content')])

OUT = Path('/content/work/ext_crop'); OUT.mkdir(parents=True, exist_ok=True)
print('출력:', OUT)


In [ ]:
# == §1 크롭 함수 (학습·추론 공용) + 검증 ==
# ★이 함수 하나만 쓴다. 학습·추론이 다른 전처리를 쓰면 CT에서 겪은 해상도 불일치가 재현된다.
PAD      = 0.10     # 0729 §4에서 확정 (결함 보존 Damaged 100% / Pollution 99.7%)
SAT_TH   = 40       # 채도 임계 — 배경=무채색, 셀=유채색
DENS_TH  = 0.10     # 행·열 마스크 밀도 임계 (흩어진 노이즈에 강건)
MIN_AREA, MAX_AREA = 0.02, 0.90   # 추정 면적이 이 범위 밖이면 실패로 보고 원본 사용

def cell_bbox(im, small=400):
    """원본 PIL 이미지 → 셀 영역 bbox(normalized) 또는 None(추정 실패).
       배경이 무채색이라는 전제. 실패 시 호출부가 원본 전체로 폴백한다."""
    t = im.copy(); t.thumbnail((small, small))
    hsv = np.asarray(t.convert('HSV'))
    S = hsv[:, :, 1].astype(np.int16)
    m = S > SAT_TH
    if m.mean() < 0.01 or m.mean() > 0.98:          # 채도로 안 갈리면 밝기로
        V = hsv[:, :, 2].astype(np.int16)
        m = np.abs(V - np.median(np.concatenate([V[0], V[-1]]))) > 40
    if m.sum() < 50: return None
    h, w = m.shape; rs, cs = m.sum(1), m.sum(0)
    if rs.max() == 0 or cs.max() == 0: return None
    ry = np.where(rs > rs.max()*DENS_TH)[0]; cx = np.where(cs > cs.max()*DENS_TH)[0]
    if len(ry) == 0 or len(cx) == 0: return None
    bb = (cx.min()/w, ry.min()/h, (cx.max()+1)/w, (ry.max()+1)/h)
    a = (bb[2]-bb[0])*(bb[3]-bb[1])
    return None if not (MIN_AREA <= a <= MAX_AREA) else bb

def crop_box(im, pad=PAD):
    """→ (x1,y1,x2,y2) 정규화 크롭 범위, ok(추정 성공 여부).
       실패 시 원본 전체(0,0,1,1)를 돌려주므로 파이프라인이 멈추지 않는다."""
    bb = cell_bbox(im)
    if bb is None: return (0.0, 0.0, 1.0, 1.0), False
    x1, y1, x2, y2 = bb; bw, bh = x2-x1, y2-y1
    return (max(0.0, x1-bw*pad/2), max(0.0, y1-bh*pad/2),
            min(1.0, x2+bw*pad/2), min(1.0, y2+bh*pad/2)), True

def apply_crop(im, box):
    W, H = im.size
    return im.crop((int(box[0]*W), int(box[1]*H), int(box[2]*W), int(box[3]*H)))

def to_crop_coords(pts, box):
    """원본 정규화 좌표 → 크롭 정규화 좌표. 범위 밖은 클리핑."""
    bw, bh = box[2]-box[0], box[3]-box[1]
    if bw <= 0 or bh <= 0: return None
    out = [(min(1.0, max(0.0, (x-box[0])/bw)), min(1.0, max(0.0, (y-box[1])/bh))) for x, y in pts]
    xs = [p[0] for p in out]; ys = [p[1] for p in out]
    if max(xs)-min(xs) < 1e-4 or max(ys)-min(ys) < 1e-4: return None   # 완전히 밖
    return out

def to_orig_coords(pts, box):
    """★크롭 좌표 → 원본 좌표 (리포트는 원본 기준이어야 함)."""
    bw, bh = box[2]-box[0], box[3]-box[1]
    return [(box[0]+x*bw, box[1]+y*bh) for x, y in pts]

# ── 왕복 변환 자체 검증 ───────────────────────────────────────────────────
_b = (0.3, 0.1, 0.7, 0.9); _p = [(0.4, 0.2), (0.6, 0.5), (0.35, 0.85)]
_rt = to_orig_coords(to_crop_coords(_p, _b), _b)
assert all(abs(a[0]-b[0]) < 1e-9 and abs(a[1]-b[1]) < 1e-9 for a, b in zip(_p, _rt)), _rt
print('✅ 좌표 왕복 변환 검증 통과 (원본 → 크롭 → 원본 일치)')

# ── 합성 이미지로 추정 검증 ───────────────────────────────────────────────
_im = Image.new('RGB', (1920, 1080), (245, 242, 238))
ImageDraw.Draw(_im).rectangle([700, 60, 1180, 1010], fill=(30, 60, 180))
_bx, _ok = crop_box(_im)
print(f'✅ 합성 검증: 추정 {[round(v,3) for v in _bx]} ok={_ok} '
      f'(실제 셀 {700/1920:.3f},{60/1080:.3f},{1180/1920:.3f},{1010/1080:.3f})')
assert _ok and abs(_bx[0]-700/1920) < 0.05


In [ ]:
# == §1.5 zip 인덱스 + 크롭 사전점검 (§2 선행 · 삭제 금지) ==
# ★★ §2가 이 셀의 MEMBER · LBLMEM_DET · LBLMEM_SEG · zread 를 쓴다.
#    '일회성 점검'처럼 보이지만 **인덱스 생성이 본체**다. 지우면 §2가 NameError로 죽는다.
!pip -q install pandas
_need = [n for n in ('ZIPS', 'EXT', 'crop_box', 'apply_crop') if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

N_CHECK   = 200      # 점검 표본
NPROC_HINT = max(1, (os.cpu_count() or 4) - 1)   # 빌드 시간 추정용
FAIL_GATE = 0.05     # 폴백률이 이 값을 넘으면 임계 재조정 후 다시 (빌드로 넘어가지 말 것)
IMGX = {'.jpg', '.jpeg', '.png', '.bmp'}

# ── zip 멤버 인덱스 (§2에서 재사용) ───────────────────────────────────────
print('zip 인덱스 작성 중...')
MEMBER, LBLMEM_DET, LBLMEM_SEG = {}, {}, {}
for z in ZIPS:
    try: names = zipfile.ZipFile(z).namelist()
    except Exception as e: print('  스킵', z.name, str(e)[:60]); continue
    for n in names:
        p = Path(n); sfx = p.suffix.lower()
        if sfx in IMGX: MEMBER.setdefault(p.name, (z, n))
        elif sfx == '.txt' and 'label' in p.parent.name.lower():
            d = p.parent.name.lower()
            # 폴더명이 labels_det/labels_seg가 아니라 그냥 labels 일 수 있다.
            # parse_label이 두 포맷을 모두 읽으므로 그때는 양쪽에 넣어둔다.
            if 'det' in d: LBLMEM_DET.setdefault(p.stem, (z, n))
            elif 'seg' in d: LBLMEM_SEG.setdefault(p.stem, (z, n))
            else: LBLMEM_DET.setdefault(p.stem, (z, n)); LBLMEM_SEG.setdefault(p.stem, (z, n))
    print(f'  {z.name}: 이미지 {len(MEMBER):,} / det라벨 {len(LBLMEM_DET):,} / seg라벨 {len(LBLMEM_SEG):,} (누적)')
assert MEMBER, '★zip에서 이미지를 못 찾음'
assert LBLMEM_DET or LBLMEM_SEG, '★zip에서 라벨(.txt)을 못 찾음 — 폴더 구조 확인'
if LBLMEM_DET is not LBLMEM_SEG and not LBLMEM_DET:
    print('  ⚠️ det 라벨이 없다 → §2의 TASK를 "seg"로. 단 Damaged 이미지 11%를 잃는다')
_shared = set(LBLMEM_DET) == set(LBLMEM_SEG) and bool(LBLMEM_DET)
if _shared: print('  ℹ️ 라벨 폴더가 det/seg로 안 나뉘어 있다 — 포맷은 파서가 자동 판별')

_ZF = {}
def zread(z, n):
    if z not in _ZF: _ZF[z] = zipfile.ZipFile(z)
    return _ZF[z].read(n)

# ── 표본 크롭 점검 ────────────────────────────────────────────────────────
import random as _rk
_rk.seed(42)
pool = [r for r in EXT.itertuples() if r.output_image_name in MEMBER]
samp = _rk.sample(pool, min(N_CHECK, len(pool)))
print(f'\n표본 {len(samp)}장 점검 중...')

rec = []; nbytes = 0; t_read = t_crop = 0.0
for r in samp:
    z, mem = MEMBER[r.output_image_name]
    _t = time.time()
    try: raw = zread(z, mem)
    except Exception: rec.append((r, None, False, None)); continue
    t_read += time.time()-_t; nbytes += len(raw)
    _t = time.time()
    try: im = Image.open(_io.BytesIO(raw)).convert('RGB')
    except Exception: rec.append((r, None, False, None)); continue
    box, ok = crop_box(im)
    t_crop += time.time()-_t
    a = (box[2]-box[0])*(box[3]-box[1])
    rec.append((r, box, ok, a))

good = [x for x in rec if x[1] is not None]
fail = [x for x in good if not x[2]]
areas = np.array([x[3] for x in good if x[2]])
print(f'\n■ 크롭 사전점검 결과')
print(f'  읽기 실패        {len(rec)-len(good)}')
print(f'  ★폴백(원본 사용)  {len(fail)}/{len(good)} = {len(fail)/max(1,len(good)):.1%}   (게이트 {FAIL_GATE:.0%})')
if len(areas):
    print(f'  크롭 면적비      중앙 {np.median(areas):.3f}  p05 {np.percentile(areas,5):.3f}  p95 {np.percentile(areas,95):.3f}')
    print(f'  선형 배율        중앙 {1/np.sqrt(np.median(areas)):.2f}x  (이전 측정 1.98x)')

# 폴백이 특정 그룹에 쏠리는지 (셀 형태·application은 메타데이터일 뿐이나 진단엔 유용)
if fail:
    from collections import Counter as _C
    print(f'  폴백 쏠림(application): {dict(_C(x[0].application for x in fail))}')
    print(f'  폴백 쏠림(split)      : {dict(_C(x[0].split_role for x in fail))}')

# ── 게이트 ────────────────────────────────────────────────────────────────
fr = len(fail)/max(1, len(good))
print('\n▶ ' + ('통과 — §2 빌드로 진행' if fr <= FAIL_GATE else
      f'★중단 — 폴백률 {fr:.1%} > {FAIL_GATE:.0%}. §1의 SAT_TH/DENS_TH/MIN_AREA 조정 후 재실행'))
assert fr <= FAIL_GATE, f'폴백률 {fr:.1%}가 게이트 {FAIL_GATE:.0%} 초과'
print(f'  인덱스 준비 완료 — 이미지 {len(MEMBER):,} · det라벨 {len(LBLMEM_DET):,} '
      f'· seg라벨 {len(LBLMEM_SEG):,}  (§2가 이걸 쓴다)')


In [ ]:
# == §2 데이터셋 빌드 — 순차 리더 + 병렬 크롭 ==
#   읽기는 메인이 오프셋 순으로 혼자, 크롭·저장만 워커가 한다 (워커는 zip을 안 만진다).
#
# v4.2 반영
#   ① TASK='det'  — seg 제외분 8,775장 중 50.9%가 Damaged. 제품은 위치 리포트라 마스크 불필요
#   ② 3분할 서브샘플 — train 무결함 32.9%. 배경 54%가 recall을 눌렀던 실측 반영(→8.9%)
!pip -q install pandas
import multiprocessing as _mp, gc
_need = [n for n in ('OUT', 'EXT', 'MEMBER', 'crop_box', 'to_crop_coords', 'zread')
         if n not in globals()]
assert not _need, f'★§1.5를 먼저 실행하세요 — 없는 변수: {_need}'

TASK          = 'det'    # 'det'(권장) | 'seg'
MAX_PER_SPLIT = None     # 시험 실행은 500 등. None=전량
DMG_KEEP      = 1.00     # ID cap 적용 후의 비율. 보통 1.00으로 두고 cap으로 조절한다
POLL_KEEP     = 0.20     # Pollution만 있는 이미지 — recall이 이미 1.000이라 줄여도 안 떨어짐
# ★★ ID당 cap — 매니페스트 분석의 결론
#   → 셀 수는 그대로 두고 셀당 반복만 줄인다. 정보는 안 버린다.
DMG_ID_CAP    = 40       # None이면 무제한(구 동작). 40 → Damaged 40,524 → 10,126장
POLL_ID_CAP   = 40       # Pollution만 있는 이미지도 같은 편중이 있다(ID당 p50 184)
CLEAN_KEEP    = 0.10     # 무결함 이미지 — 배경 비율을 ~9%로 (YOLO 권장선)
# ★무결함 배경을 뽑는 순서 — 매니페스트 분석
CLEAN_PREFER_APP = '가전'   # None이면 구 동작(무작위)
VAL_CAP       = 4000     # val은 매 에폭 돌므로 상한. 균등 샘플이라 분포는 보존
JPEG_Q        = 92
NPROC         = max(1, (os.cpu_count() or 4) - 1)
BLOCK         = 1000     # 한 번에 메모리에 올리는 장수 (1000 x 0.53MB ~ 0.5GB)

LBLMEM = LBLMEM_DET if TASK == 'det' else LBLMEM_SEG
assert LBLMEM, f'★{TASK} 라벨이 zip에 없다 — TASK를 바꾸거나 데이터팀에 요청'
SEL = EXT[EXT[f'included_{TASK}'].str.lower() == 'true']
for sub in ('images', 'labels'):
    for sp in ('train', 'val', 'test'): (OUT/sub/sp).mkdir(parents=True, exist_ok=True)

def _tf(v): return str(v).lower() in ('true', '1', 'y', 'yes')

def parse_label(raw):
    """seg 폴리곤과 det bbox를 모두 받는다 → (class, [(x,y), ...])"""
    out = []
    for ln in raw.decode('utf-8', 'ignore').splitlines():
        v = ln.split()
        if len(v) < 5: continue
        c = int(float(v[0])); n = list(map(float, v[1:]))
        if len(n) == 4:                                   # det: cx cy w h → 네 꼭짓점
            cx, cy, w, h = n
            pts = [(cx-w/2, cy-h/2), (cx+w/2, cy-h/2), (cx+w/2, cy+h/2), (cx-w/2, cy+h/2)]
        else:
            pts = list(zip(n[0::2], n[1::2]))
        out.append((c, pts))
    return out

def emit_label(c, pts):
    """TASK에 맞는 한 줄. det면 min/max bbox, seg면 폴리곤 그대로."""
    if TASK == 'det':
        xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
        x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
        return f'{c} {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} {x2-x1:.6f} {y2-y1:.6f}'
    return str(c) + ' ' + ' '.join(f'{x:.6f} {y:.6f}' for x, y in pts)

def crop_one(payload):
    """워커: 바이트를 받아 크롭·저장. zip은 건드리지 않는다."""
    raw, lbl_raw, stem, ysp = payload
    st = Counter()
    try: im = Image.open(_io.BytesIO(raw)).convert('RGB')
    except Exception: st['이미지 디코드 실패'] += 1; return st
    box, ok = crop_box(im)
    st['크롭 성공' if ok else '★폴백(원본 사용)'] += 1
    ci = apply_crop(im, box)
    if min(ci.size) < 16: st['크롭 너무 작음(원본 사용)'] += 1; box, ci = (0., 0., 1., 1.), im

    lines = []
    if lbl_raw:
        for c, pts in parse_label(lbl_raw):
            q = to_crop_coords(pts, box)
            if q is None: st['라벨 크롭 밖 제외'] += 1; continue
            lines.append(emit_label(c, q)); st['라벨 유지'] += 1
    ci.save(OUT/'images'/ysp/f'{stem}.jpg', quality=JPEG_Q)
    (OUT/'labels'/ysp/f'{stem}.txt').write_text('\n'.join(lines))
    st['이미지 저장'] += 1
    return st

# ── 작업 목록 (여기서 믹스가 정해진다) ────────────────────────────────────
import random as _rk
JOBS = []
print(f'TASK={TASK} | 라벨 {len(LBLMEM):,}개 | 워커 {NPROC}\n')
for ysp in ('train', 'val', 'test'):
    rows = [r for r in SEL[SEL['split_role'] == ysp].itertuples()
            if r.output_image_name in MEMBER]
    n0 = len(rows)
    if ysp == 'train':                                   # ★train만 클래스별 서브샘플
        _rk.seed(42)
        dmg   = [r for r in rows if _tf(r.has_damaged)]
        poll  = [r for r in rows if not _tf(r.has_damaged) and _tf(r.has_pollution)]
        clean = [r for r in rows if not _tf(r.has_damaged) and not _tf(r.has_pollution)]
        def _cap(lst, cap):
            """ID당 최대 cap장. 셀 종류는 다 살리고 셀당 중복만 자른다."""
            if not cap: return lst
            by = defaultdict(list)
            for r in lst: by[r.battery_id].append(r)
            out = []
            for k in sorted(by):                      # 정렬 = 재현성
                v = by[k]; _rk.shuffle(v); out += v[:cap]
            return out
        if CLEAN_PREFER_APP:                       # 가전 먼저, 부족분만 산업
            _pref = [r for r in clean if str(getattr(r, 'application', '')) == CLEAN_PREFER_APP]
            _rest = [r for r in clean if str(getattr(r, 'application', '')) != CLEAN_PREFER_APP]
            _rk.shuffle(_pref); _rk.shuffle(_rest)
            clean = _pref + _rest                  # 앞에서부터 뽑히므로 가전이 우선
            _npref = len(_pref)
        _d0, _p0 = len(dmg), len(poll)
        _nid = len({r.battery_id for r in dmg})
        dmg  = _cap(dmg,  DMG_ID_CAP)
        poll = _cap(poll, POLL_ID_CAP)
        rows = (_rk.sample(dmg,   int(len(dmg)  *DMG_KEEP)) +
                _rk.sample(poll,  int(len(poll) *POLL_KEEP)) +
                clean[:int(len(clean)*CLEAN_KEEP)])   # ★앞에서 자른다 = 가전 우선
        _c = int(len(clean)*CLEAN_KEEP)
        print(f'[train] Damaged {int(len(dmg)*DMG_KEEP):,}/{_d0:,} (ID {_nid}개, cap {DMG_ID_CAP}) · '
              f'Pollution만 {int(len(poll)*POLL_KEEP):,}/{_p0:,} · '
              f'무결함 {_c:,}/{len(clean):,}  → 총 {len(rows):,} (배경 {_c/max(1,len(rows)):.1%})')
        print(f'        셀당 노출 {_d0/max(1,_nid):.0f}장 → {len(dmg)/max(1,_nid):.0f}장 '
              f'(다양성 {_nid}셀 유지, 암기 압력 {len(dmg)/max(1,_d0):.0%})')
        if CLEAN_PREFER_APP:
            _got = sum(1 for r in clean[:_c]
                       if str(getattr(r, 'application', '')) == CLEAN_PREFER_APP)
            print(f'        무결함 배경: {CLEAN_PREFER_APP} {_got:,}/{_c:,} ({_got/max(1,_c):.0%}) '
                  f'· 가전 풀 {_npref:,}장')
    elif ysp == 'val' and VAL_CAP and len(rows) > VAL_CAP:
        _rk.seed(42); rows = _rk.sample(rows, VAL_CAP)   # 균등 — 분포 보존
        print(f'[val]   {n0:,} → {len(rows):,} (균등 샘플, 매 에폭 돌므로 상한)')
    else:
        print(f'[{ysp:5s}] {len(rows):,} (전량)')
    if MAX_PER_SPLIT:
        _rk.seed(42); _rk.shuffle(rows)      # ★섞고 자른다 — 안 그러면 Damaged만 뽑혀
        rows = rows[:MAX_PER_SPLIT]          #   시험실행 소요시간이 대표성을 잃는다
    JOBS += [(r.output_image_name, r.output_label_stem, ysp) for r in rows]

# ── 오프셋 인덱스 (이미지·라벨 둘 다) ─────────────────────────────────────
if 'OFFSET' not in globals(): OFFSET = {}
OFFL = {}
print('\n오프셋 인덱스...', flush=True)
for z in ZIPS:
    for zi in zipfile.ZipFile(z).infolist():
        p = Path(zi.filename)
        if p.name in MEMBER and MEMBER[p.name][0] == z: OFFSET[p.name] = zi.header_offset
        st_ = LBLMEM.get(p.stem)
        if st_ and st_[1] == zi.filename: OFFL[p.stem] = zi.header_offset
print(f'  이미지 {len(OFFSET):,} / 라벨 {len(OFFL):,}')

# ── 큰 객체 정리 후 워커 생성 (fork 시점 메모리를 최소로) ─────────────────
for _n in ('SEL', 'rows', 'dmg', 'poll', 'clean', '_pool', 'rec', 'samp',
           'good', 'fail', 'names_sorted', 'par', 'seq'):
    globals().pop(_n, None)
gc.collect()

stat = Counter(); t0 = time.time()
print(f'\n총 {len(JOBS):,}장 빌드 시작 (순차 리더 + 워커 {NPROC})', flush=True)

with _mp.Pool(NPROC) as pool:
    # ── 1패스: 라벨 일괄 선적재 (작아서 전부 메모리에 든다) ──
    need = sorted({j[1] for j in JOBS} & set(LBLMEM),
                  key=lambda s_: (str(LBLMEM[s_][0]), OFFL.get(s_, 0)))
    LBLRAW = {}
    _t = time.time()
    for i, s_ in enumerate(need):
        LBLRAW[s_] = zread(*LBLMEM[s_])
        if (i+1) % 20000 == 0: print(f'  라벨 {i+1:,}/{len(need):,}', flush=True)
    print(f'  라벨 {len(LBLRAW):,}개 적재 {time.time()-_t:.0f}초 '
          f'({sum(map(len, LBLRAW.values()))/1e6:.0f} MB)', flush=True)

    # ── 2패스: 이미지를 오프셋 순으로 읽어 블록 단위로 워커에 넘긴다 ──
    JOBS.sort(key=lambda j: (str(MEMBER[j[0]][0]), OFFSET.get(j[0], 0)))
    done = 0
    for b0 in range(0, len(JOBS), BLOCK):
        blk = JOBS[b0:b0+BLOCK]
        payload = []
        for img, lbl, ysp in blk:                       # ★메인이 혼자 순차로 읽는다
            try: raw = zread(*MEMBER[img])
            except Exception: stat['이미지 읽기 실패'] += 1; continue
            payload.append((raw, LBLRAW.get(lbl), Path(img).stem, ysp))
        for st in pool.imap_unordered(crop_one, payload, chunksize=16): stat += st
        done += len(blk); del payload
        el = time.time()-t0
        print(f'  {done:,}/{len(JOBS):,}  {el/60:.1f}분  {done/el:.0f}장/s  '
              f'남은 ~{(len(JOBS)-done)/max(done/el,1e-9)/60:.0f}분', flush=True)

print('\n■ 빌드 결과')
for k, v in stat.most_common(): print(f'  {k:<24} {v:,}')
fail = stat['★폴백(원본 사용)']; tot = stat['크롭 성공']+fail
print(f'  크롭 폴백률: {fail:,}/{tot:,} = {fail/max(1,tot):.2%}')
sz = sum(f.stat().st_size for f in (OUT/'images').rglob('*.jpg'))
print(f'  크롭본 총 용량: {sz/1e9:.2f} GB   소요 {(time.time()-t0)/60:.1f}분')
for sp in ('train', 'val', 'test'):
    print(f'  {sp:5s} {len(list((OUT/"images"/sp).glob("*.jpg"))):,}장')


In [ ]:
# == §2.5 빌드 검증 + Drive 백업 (학습 전 필수, ~3분) ==
# ★클래스 인덱스를 한 번도 확인한 적이 없다. names=['Damaged','Pollution']이 0=Damaged를
#   전제하는데 뒤집혀 있으면 학습은 멀쩡히 되고 **리포트에서 두 클래스가 서로 바뀐다.**
#   매니페스트가 알려주는 비율(Damaged:Pollution = 1:11)로 즉시 판별한다.
!pip -q install pandas
from collections import Counter
from pathlib import Path
import numpy as np, subprocess, time
_need = [n for n in ('OUT',) if n not in globals()]
assert not _need, f'★§2를 먼저 실행하세요 — 없는 변수: {_need}'

BACKUP_TRAIN = True     # False면 val+test만 백업(eval에 필요한 최소분, ~1.7GB)
NAMES_GUESS  = ['Damaged', 'Pollution']      # §3의 names와 같아야 한다

# ── 1. 라벨 통계 ─────────────────────────────────────────────────────────
print('■ 라벨 검증')
tot = Counter(); empty = {}; nfile = {}; bad_range = bad_ncol = 0
wh = []
for sp in ('train', 'val', 'test'):
    c = Counter(); e = 0; fs = sorted((OUT/'labels'/sp).glob('*.txt'))
    nfile[sp] = len(fs)
    for f in fs:
        t = f.read_text().strip()
        if not t: e += 1; continue
        for ln in t.splitlines():
            v = ln.split()
            if len(v) != 5: bad_ncol += 1; continue
            k = int(float(v[0])); c[k] += 1
            n = list(map(float, v[1:]))
            if any(x < -1e-6 or x > 1+1e-6 for x in n): bad_range += 1
            if sp == 'train' and len(wh) < 200000: wh.append((n[2], n[3]))
    empty[sp] = e; tot += c
    print(f'  {sp:5s} 라벨파일 {len(fs):>7,}  빈파일 {e:>6,} ({e/max(1,len(fs)):5.1%})  '
          + ' '.join(f'cls{k} {v:,}' for k, v in sorted(c.items())))

# ── 2. ★클래스 인덱스 판별 ───────────────────────────────────────────────
n0, n1 = tot.get(0, 0), tot.get(1, 0)
print(f'\n■ 클래스 인덱스 판별   cls0 {n0:,} : cls1 {n1:,}  =  1:{n1/max(1,n0):.1f}')
print('  매니페스트 실측: Damaged 106,043 : Pollution 1,170,102 = 1:11.0 (train 전량)')
if n0 and n1:
    if n0 < n1:
        print(f'  ✅ cls0이 소수 → cls0=Damaged, cls1=Pollution. names={NAMES_GUESS} 맞다')
        assert NAMES_GUESS[0] == 'Damaged', '★names 순서가 반대다'
    else:
        print(f'  🔴 cls0이 다수 → cls0=Pollution이다! §3의 names를 뒤집을 것:')
        print("     names = ['Pollution', 'Damaged']")
        raise AssertionError('클래스 인덱스가 예상과 반대 — names 수정 후 재실행')
else:
    raise AssertionError(f'★한 클래스가 0건이다 (cls0={n0}, cls1={n1}) — 라벨 변환 실패')

# ── 3. 무결성 ────────────────────────────────────────────────────────────
print('\n■ 무결성')
ok = True
for sp in ('train', 'val', 'test'):
    ni = len(list((OUT/'images'/sp).glob('*.jpg')))
    m = '✅' if ni == nfile[sp] else '🔴'
    if ni != nfile[sp]: ok = False
    print(f'  {sp:5s} 이미지 {ni:>7,} / 라벨 {nfile[sp]:>7,}  {m}')
print(f'  좌표 범위 이탈 {bad_range:,}  {"✅" if bad_range == 0 else "🔴"}')
print(f'  컬럼수 오류   {bad_ncol:,}  {"✅" if bad_ncol == 0 else "🔴"}  (det는 5개여야 함)')
assert ok and bad_range == 0 and bad_ncol == 0, '★무결성 실패 — 학습 금지'

# 배경(빈 라벨) 비율 = recall 억제 여부
bg = empty['train']/max(1, nfile['train'])
print(f'  train 배경 비율 {bg:.1%}  ' +
      ('✅ (BG 54%가 recall을 눌렀던 실측 대비 안전권)' if bg < 0.15 else '🔴 너무 높다'))

# 결함 크기 (크롭 효과 확인)
if wh:
    a = np.array(wh); px = np.maximum(a[:, 0]*520, a[:, 1]*980)   # 크롭본 대략 크기
    print(f'\n■ 결함 긴변(px, 크롭본 기준 추정)  p10 {np.percentile(px,10):.0f}  '
          f'중앙 {np.median(px):.0f}  p90 {np.percentile(px,90):.0f}  |  32px 미만 {np.mean(px<32):.0%}')
    print('  참고: 원본 기준으로는 중앙 14.5x15.7px였다 (매니페스트 EXT_train)')

# ── 4. Drive 백업 ────────────────────────────────────────────────────────
# 세션이 죽으면 23분을 다시 쓴다. eval은 다른 세션이라 어차피 필요하다.
# ★파일명에 v42 — eval §0이 CROP_TAG로 구 ext_crop_test.zip을 걸러낸다.
DST = Path('/content/drive/MyDrive/kt_out'); DST.mkdir(parents=True, exist_ok=True)
parts = ['images/val', 'labels/val', 'images/test', 'labels/test'] + \
        (['images/train', 'labels/train'] if BACKUP_TRAIN else [])
zp = DST/'ext_crop_v42.zip'
print(f'\n■ Drive 백업 → {zp}')
print(f'  대상: {parts}')
t0 = time.time()
# -0 = 무압축. JPEG는 어차피 안 줄고 CPU만 먹는다.
r = subprocess.run(['zip', '-0', '-q', '-r', str(zp)] + parts, cwd=str(OUT),
                   capture_output=True, text=True)
if r.returncode != 0:
    print('  🔴 zip 실패:', r.stderr[:300])
else:
    print(f'  ✅ {zp.stat().st_size/1e9:.2f} GB  {(time.time()-t0)/60:.1f}분')
    print(f'  eval 세션에서 자동으로 찾는다 (CROP_TAG=\'v42\')')

print('\n' + '#'*70)
print('▶ 검증 통과 — §3 학습으로 진행')


In [ ]:
# == §2.6 세션 복구 + ID cap 적용 (재크롭 불필요, ~6분) ==
# ★세션이 죽어 /content/work/ext_crop 이 날아갔을 때 쓴다.
#   Drive의 빌드본 zip(ext_crop_v42.zip)을 풀고, **파일을 쳐내서** cap을 적용한다.
#   cap 판단에 필요한 정보는 전부 **라벨 파일과 파일명**에 있다(매니페스트 불필요):
#     · 클래스   : 라벨 첫 칸 (0=Damaged, 1=Pollution), 빈 파일 = 무결함
#     · battery_id: RGB_cell_<form>_<battery_id>_<frame>
import subprocess, shutil, random, re as _re, time
from pathlib import Path
from collections import defaultdict, Counter

OUT = Path('/content/work/ext_crop')
# ★이 셀은 빠른 경로(§0→§2.6→§2.9→§2.95→§3)의 진입점이다. §2를 건너뛰므로
#   §2가 정의하던 TASK를 여기서 확정해야 뒤 셀들이 산다.
TASK = globals().get('TASK', 'det')
DMG_ID_CAP  = globals().get('DMG_ID_CAP', 40)
POLL_ID_CAP = globals().get('POLL_ID_CAP', 40)
CLEAN_KEEP  = globals().get('CLEAN_KEEP', 0.10)
CLEAN_PREFER_APP = globals().get('CLEAN_PREFER_APP', '가전')
SEED = 42

# ── 1. 복구 ──────────────────────────────────────────────────────────────
if not (OUT/'images'/'train').exists() or not any((OUT/'images'/'train').glob('*.jpg')):
    zs = sorted(Path('/content/drive/MyDrive').rglob('*crop*v42*.zip')) + \
         sorted(Path('/content/drive/MyDrive').rglob('*crop*.zip'))
    assert zs, '★Drive에서 크롭본 zip을 못 찾음 — §1.5·§2로 재빌드해야 한다'
    z = zs[0]; OUT.mkdir(parents=True, exist_ok=True)
    print(f'해제: {z.name} ({z.stat().st_size/1e6:.0f} MB) → {OUT}', flush=True)
    t0 = time.time()
    subprocess.run(['unzip', '-q', '-o', str(z), '-d', str(OUT)], capture_output=True)
    if not (OUT/'images'/'train').exists():
        for sub in OUT.rglob('images/train'):
            OUT = sub.parent.parent; break
    print(f'  {(time.time()-t0)/60:.1f}분')
assert (OUT/'images'/'train').exists(), (
    '★zip에 train이 없다(백업이 val+test만이었다) → §1.5·§2로 재빌드해야 한다')
n0 = len(list((OUT/'images'/'train').glob('*.jpg')))
print(f'복구 완료: train {n0:,}장  ·  TASK={TASK}  ·  OUT={OUT}')

# ── 2. 매니페스트 조회 (파일명 파싱 안 한다) ────────────────────────────
# ★battery_id·클래스·application 전부 데이터팀이 준 매니페스트에 있다.
assert 'EXT' in globals(), '★§0을 먼저 실행하세요 (매니페스트 EXT가 필요)'
_tf = lambda v: str(v).lower() in ('true', '1', 'y', 'yes')
M = {}
for r in EXT.itertuples():
    M[str(r.output_label_stem)] = (str(r.battery_id), _tf(r.has_damaged),
                                   _tf(r.has_pollution), str(getattr(r, 'application', '')))

stems = [p.stem for p in sorted((OUT/'labels'/'train').glob('*.txt'))]
miss = [x for x in stems if x not in M]
print(f'  매니페스트 조회: {len(stems):,}장 중 {len(stems)-len(miss):,}건 적중 '
      f'({1-len(miss)/max(1,len(stems)):.1%})')
assert len(miss) < len(stems)*0.01, (
    f'★매니페스트에 없는 stem이 {len(miss):,}건 — 크롭본과 매니페스트가 다른 버전이다. '
    f'예: {miss[:2]}')

dmg  = [x for x in stems if x in M and M[x][1]]
poll = [x for x in stems if x in M and not M[x][1] and M[x][2]]
clean = [x for x in stems if x in M and not M[x][1] and not M[x][2]]
bid = lambda x: M[x][0]
print(f'  Damaged {len(dmg):,} · Pollution만 {len(poll):,} · 무결함 {len(clean):,}  '
      f'(ID {len({bid(x) for x in stems if x in M})}개)')
assert dmg and poll, '★분류 결과가 비었다'

# ── 3. ID cap ────────────────────────────────────────────────────────────
def cap(xs, k):
    if not k: return set(xs)
    by = defaultdict(list)
    for x in xs: by[bid(x)].append(x)
    rk = random.Random(SEED); out = []
    for key in sorted(by):
        v = sorted(by[key]); rk.shuffle(v); out += v[:k]
    return set(out)

_d, _p = cap(dmg, DMG_ID_CAP), cap(poll, POLL_ID_CAP)
keep = _d | _p
_nid = len({bid(x) for x in dmg})
print(f'  cap {DMG_ID_CAP}: Damaged {len(dmg):,}→{len(_d):,} (ID {_nid}개 유지, '
      f'셀당 {len(dmg)/max(1,_nid):.0f}→{len(_d)/max(1,_nid):.0f}장) · '
      f'Pollution만 {len(poll):,}→{len(_p):,}')

# 무결함 배경 — 가전 우선(매니페스트 application)
n_clean = int(len(clean)*CLEAN_KEEP)
rk = random.Random(SEED); cl = sorted(clean); rk.shuffle(cl)
if CLEAN_PREFER_APP:
    pref = [x for x in cl if M[x][3] == CLEAN_PREFER_APP]
    cl = pref + [x for x in cl if M[x][3] != CLEAN_PREFER_APP]
    print(f'  무결함 배경 {n_clean:,}장 — {CLEAN_PREFER_APP} 풀 {len(pref):,}장에서 우선')
keep |= set(cl[:n_clean])

# ── 4. 삭제 ──────────────────────────────────────────────────────────────
gone = 0
for ip in sorted((OUT/'images'/'train').glob('*.jpg')):
    if ip.stem in keep: continue
    ip.unlink(missing_ok=True)
    (OUT/'labels'/'train'/f'{ip.stem}.txt').unlink(missing_ok=True)
    gone += 1
for c in OUT.rglob('*.cache'): c.unlink()        # 라벨 캐시 무효화 (안 지우면 옛 목록을 쓴다)
n1 = len(list((OUT/'images'/'train').glob('*.jpg')))
print(f'\n삭제 {gone:,}장 → train {n0:,} → {n1:,}장 ({n1/max(1,n0):.0%})')
assert n1 == len(list((OUT/'labels'/'train').glob('*.txt'))), '★이미지·라벨 짝이 안 맞는다'
print(f'  val {len(list((OUT/"images"/"val").glob("*.jpg"))):,} · '
      f'test {len(list((OUT/"images"/"test").glob("*.jpg"))):,} (건드리지 않음)')
print('\n▶ §2.9로 라벨 검사 후 §3 학습')


In [ ]:
# == §2.9 학습 직전 최종 검사 — 라벨이 진짜 붙는가 (30초, 사고 대응) ==
# ★왜 필요한가: ultralytics는 라벨 파일을 못 찾아도 에러를 내지 않는다. 그 이미지를 배경으로
#   취급하고 조용히 계속 돈다 → 전부 배경 = 모델이 "아무것도 없다"를 배운다.
#   → **ultralytics와 똑같은 경로 규칙**으로 라벨을 찾아보고, 못 찾으면 여기서 멈춘다.
from pathlib import Path
import os, glob as _g, yaml
from collections import Counter

_out = Path(globals().get('OUT', '/content/work/ext_crop'))
_names = ['defect'] if globals().get('SINGLE_CLASS', False) else ['Damaged', 'Pollution']
YML = Path(globals().get('yml') or (_out/'data.yaml'))
if not YML.exists():
    assert (_out/'images'/'train').exists(), f'★크롭 데이터셋이 없다: {_out} — §2.6 또는 §2 먼저'
    YML.write_text(yaml.safe_dump({'path': str(_out), 'train': 'images/train',
                                   'val': 'images/val', 'names': _names},
                                  allow_unicode=True, sort_keys=False), encoding='utf-8')
    print(f'data.yaml 생성: {YML}  names={_names}')
yml = YML                                   # §2.95·§3이 그대로 쓴다
cfg = yaml.safe_load(YML.read_text())
root = Path(cfg['path'])
print(f'data.yaml : {YML}')
print(f'  path    : {root}   (존재 {root.exists()})')
print(f'  names   : {cfg["names"]}  → nc={len(cfg["names"])}')

# ── ultralytics와 동일한 규칙 ────────────────────────────────────────────
sa, sb = f'{os.sep}images{os.sep}', f'{os.sep}labels{os.sep}'
def lbl_of(p): return sb.join(str(p).rsplit(sa, 1)).rsplit('.', 1)[0] + '.txt'

bad = False
for split in ('train', 'val'):
    d = root/cfg[split]
    # ultralytics는 glob('**/*.*', recursive=True)를 쓴다. 심볼릭 링크 디렉터리는
    # 파이썬 glob의 '**'가 따라 들어가지 않는다 → 이미지 0장이 될 수 있다.
    ims = sorted(_g.glob(str(d/'**'/'*.*'), recursive=True))
    print(f'\n■ {split}')
    print(f'  이미지 디렉터리 {d}')
    print(f'    심볼릭 링크 {d.is_symlink() or d.parent.is_symlink()}  ·  '
          f'glob(**) {len(ims):,}장  ·  직접 listdir {len(list(d.glob("*.jpg"))):,}장')
    if not ims:
        print('  🔴 ultralytics가 쓰는 glob으로 이미지를 못 찾는다 (심볼릭 링크 원인).')
        bad = True; continue
    n, miss, empty = 0, 0, 0
    cls = Counter()
    for p in ims[:4000]:
        lp = Path(lbl_of(p)); n += 1
        if not lp.exists(): miss += 1; continue
        t = lp.read_text().strip()
        if not t: empty += 1; continue
        for ln in t.splitlines():
            v = ln.split()
            if len(v) >= 5: cls[int(float(v[0]))] += 1
    print(f'  표본 {n:,}장 → 라벨 없음 {miss:,} ({miss/n:.1%}) · 빈 파일 {empty:,} ({empty/n:.1%})')
    print(f'    클래스 분포 {dict(sorted(cls.items()))}  ·  장당 {sum(cls.values())/max(1,n):.1f}개')
    if miss/n > 0.5:
        print('  🔴 절반 이상이 라벨을 못 찾는다 = 전부 배경으로 학습된다. 이게 v5_1cls 사고 원인.')
        print(f'     예: {ims[0]}\n       → {lbl_of(ims[0])}')
        bad = True
    if cls and max(cls) >= len(cfg['names']):
        print(f'  🔴 라벨 클래스 {max(cls)} 가 nc={len(cfg["names"])} 를 넘는다.')
        bad = True
    if not cls:
        print('  🔴 라벨은 있는데 객체가 하나도 없다 = 전부 배경.')
        bad = True

assert not bad, '★위 문제를 고치기 전에는 학습하지 말 것 (4시간을 버린다)'
print('\n✅ 라벨이 정상으로 붙는다 — §3 학습으로 진행')


In [ ]:
# == §2.95 스모크 학습 — 진짜로 1에폭 태워본다 (90초, 학습 전 필수) ==
# ★왜 필요한가: 정적 검사는 "내 수정이 반영됐나"만 본다. 실제 학습 코드 경로를 작게 태워
#   보고, ultralytics가 dataloader에 올린 박스 수를 직접 센다. 0이면 멈춘다.
from ultralytics import YOLO
from pathlib import Path
import shutil, yaml, math, torch

# 빠른 경로에서 §2를 건너뛰어도 돌게 한다(TASK 기본 det, OUT 기본 크롭 경로)
TASK = globals().get('TASK', 'det')
OUT = Path(globals().get('OUT', '/content/work/ext_crop'))
assert (OUT/'images'/'train').exists(), f'★크롭 데이터셋 없음: {OUT} — §2.6 먼저'
YML = Path(globals().get('yml') or (OUT/'data.yaml'))
assert YML.exists(), f'★data.yaml 없음: {YML} — §2.9를 먼저 실행하면 만들어진다'
cfg = yaml.safe_load(YML.read_text())
print(f'검사 대상 data.yaml: {YML}\n  path={cfg["path"]}  names={cfg["names"]}')

N_TR, N_VA = 64, 16
SM = Path('/content/smoke')
if SM.exists(): shutil.rmtree(SM)
for sp, k in (('train', N_TR), ('val', N_VA)):
    src = Path(cfg['path'])/cfg[sp]
    (SM/'images'/sp).mkdir(parents=True); (SM/'labels'/sp).mkdir(parents=True)
    # ★결함 있는 이미지를 우선 넣는다. 배경만 뽑으면 "박스 0"이 정상처럼 보인다.
    ims = sorted(src.glob('*.jpg'))
    lab = lambda p: Path(str(p).replace('/images/', '/labels/')).with_suffix('.txt')
    withb = [p for p in ims if lab(p).exists() and lab(p).read_text().strip()]
    pick = (withb[:k] or ims[:k])
    assert pick, f'★{sp} 이미지가 없다: {src}'
    for p in pick:
        shutil.copy(p, SM/'images'/sp/p.name)
        l = lab(p)
        (SM/'labels'/sp/f'{p.stem}.txt').write_text(l.read_text() if l.exists() else '')
    print(f'  {sp}: 라벨 있는 이미지 {len(withb):,}장 중 {len(pick)}장 추출')

(SM/'data.yaml').write_text(yaml.safe_dump(
    {'path': str(SM), 'train': 'images/train', 'val': 'images/val', 'names': cfg['names']},
    allow_unicode=True, sort_keys=False), encoding='utf-8')

m = YOLO('yolo11n.pt' if TASK == 'det' else 'yolo11n-seg.pt')     # 작은 모델로 빠르게
m.train(data=str(SM/'data.yaml'), imgsz=640, batch=8, epochs=1, workers=2,
        project='/content/smoke_run', name='s', exist_ok=True,
        val=True, plots=False, verbose=False)

# ── ultralytics가 실제로 dataloader에 올린 것을 직접 센다 ────────────────
ds = m.trainer.train_loader.dataset
nb_ = sum(len(x['cls']) for x in ds.labels)
nimg = len(ds.labels)
nbg = sum(1 for x in ds.labels if len(x['cls']) == 0)
print(f'\n■ ultralytics dataloader 실측')
print(f'  이미지 {nimg}장 · 박스 {nb_:,}개 · 배경(박스0) {nbg}장 ({nbg/max(1,nimg):.0%})')
cls = {}
for x in ds.labels:
    for c in x['cls'].reshape(-1).tolist(): cls[int(c)] = cls.get(int(c), 0) + 1
print(f'  클래스 분포 {dict(sorted(cls.items()))}   (nc={len(cfg["names"])})')

import pandas as pd
rc = Path('/content/smoke_run/s/results.csv')
loss = None
if rc.exists():
    r = pd.read_csv(rc); r.columns = [c.strip() for c in r.columns]
    lc = [c for c in r.columns if 'loss' in c]
    loss = {c: float(r[c].iloc[-1]) for c in lc}
    print(f'  1에폭 loss {loss}')

print('\n■ 판정')
ok = True
if nb_ == 0:
    print('  🔴 박스 0개 — 라벨이 하나도 안 붙었다. **v5_1cls를 날린 바로 그 상태다.**')
    print('     경로 규칙(images↔labels)·심볼릭 링크·data.yaml path를 확인할 것.'); ok = False
if nbg/max(1, nimg) > 0.5:
    print(f'  🔴 배경이 {nbg/nimg:.0%} — 결함 이미지를 골라 넣었는데도 이 비율이면 라벨이 안 붙는다.'); ok = False
if cls and max(cls) >= len(cfg['names']):
    print(f'  🔴 라벨 클래스 {max(cls)} ≥ nc={len(cfg["names"])} — data.yaml names와 라벨이 불일치.'); ok = False
if loss and any(not math.isfinite(v) or v == 0 for v in loss.values()):
    print(f'  🔴 loss가 NaN/0 — 학습이 성립하지 않는다.'); ok = False
assert ok, '★스모크 실패 — 본 학습을 돌리면 4시간을 버린다. 위 원인부터 고칠 것'
print(f'  ✅ 박스 {nb_:,}개가 실제로 올라갔고 loss가 정상이다 — §3 본 학습 진행')
shutil.rmtree('/content/smoke_run', ignore_errors=True); shutil.rmtree(SM, ignore_errors=True)


In [ ]:
!pip -q install pyyaml ultralytics
# == §3 재학습 (A100/L4 권장) — 중단 시 자동 이어학습 ==
# ⚠️ 이 셀을 다시 실행하면 Drive 스냅샷을 찾아 이어서 학습한다(1에폭부터 다시 시작하지 않음).
#    처음부터 다시 하려면 CKPT 폴더와 /content/runs_main/{RUN} 을 지울 것.
from ultralytics import YOLO
import yaml, torch, shutil      # ★shutil은 아래 1클래스 빌드에서 쓴다. §0에만 있으면
                                #   커널을 새로 띄우고 이 셀만 돌릴 때 NameError로 죽는다.
# ── 선행 셀 확인 (없으면 여기서 명확히 실패) ──
TASK = globals().get('TASK', 'det')
from pathlib import Path as _P
OUT = _P(globals().get('OUT', '/content/work/ext_crop'))
assert (OUT/'images'/'train').exists(), f'★크롭 데이터셋 없음: {OUT} — §2.6 먼저'

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
# ★TASK에 맞는 사전학습 가중치. det는 seg 제외분(Damaged 50.9%)까지 쓰고 학습도 빠르다.
BASE = 'yolo11m.pt' if TASK == 'det' else 'yolo11m-seg.pt'
print('TASK:', TASK, '| base:', BASE)

IMGSZ = 1280        # v2와 동일 — 이번 판의 변수는 클래스 수 하나뿐이다
BATCH = 12
EPOCHS = 14         # ★데이터가 82,943 → ~28,800장(35%)이 되므로 에폭을 늘려 총 업데이트를
                    #   v2와 맞춘다(28.8k×14 ≈ 403k vs 82.9k×8 = 663k). 그래도 셀당 노출은
                    #   40×14=560 < 135×8=1,080 = 암기 압력 절반. 이게 이 실험의 목적이다.
RUN = 'train_ext_crop_v5_idcap'  # 이번 변수 = ID cap 하나
SINGLE_CLASS = False   # ★실험 E(1클래스)는 끈다 — §18에서 분류 이득 +0.093(🟠 최소선 미달).
                       #   cap과 같이 넣으면 v3에서 겪은 '변수 두 개' 문제가 반복된다.
                       #   cap 결과를 본 뒤 필요하면 True로.

names = ['Damaged', 'Pollution'] if not SINGLE_CLASS else ['defect']
yml = OUT/'data.yaml'
yml.write_text(yaml.safe_dump({'path': str(OUT), 'train': 'images/train', 'val': 'images/val',
                               'names': names}, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('data.yaml:', yml, '| 클래스', names)

# ── 실험 E: 1클래스 데이터셋 ─────────────────────────────────────────────
# ★원본 라벨을 덮어쓰지 않는다. 예전 코드는 in-place로 갈아엎어서 한 번 돌리면
#   2클래스 라벨이 사라졌다(되돌릴 방법 없음). 별도 디렉터리 + images는 심볼릭 링크.
if SINGLE_CLASS:
    # 🔴 심볼릭 링크를 쓰지 않는다. ultralytics는 glob('**', recursive=True)로 이미지를
    #    찾는데 파이썬 glob의 '**'는 **심볼릭 링크 디렉터리를 따라 들어가지 않는다.**
    #    대신 원본 옆에 labels_1cls를 만들고 **학습 동안만 labels와 바꿔치기**한다.
    #    복사 0회, 원본은 labels_2cls로 보존된다.
    L2, L1 = OUT/'labels_2cls', OUT/'labels_1cls'
    if not L1.exists():
        n = 0
        for sp in ('train', 'val', 'test'):
            src = OUT/'labels'/sp
            if not src.exists(): continue
            dst = L1/sp; dst.mkdir(parents=True, exist_ok=True)
            for f in src.glob('*.txt'):
                t = f.read_text().strip()
                (dst/f.name).write_text(
                    '\n'.join('0 ' + ln.split(' ', 1)[1] for ln in t.splitlines() if ln.strip()))
                n += 1
        print(f'1클래스 라벨 {n:,}개 생성 → {L1}')
    if not L2.exists():                       # 원본을 labels_2cls로 물러두고
        (OUT/'labels').rename(L2)
    if (OUT/'labels').exists() and (OUT/'labels').is_symlink():
        (OUT/'labels').unlink()
    if not (OUT/'labels').exists():           # labels 자리에 1클래스본을 놓는다
        L1.rename(OUT/'labels')
        L1 = None
    assert L2.exists(), '★2클래스 원본(labels_2cls)이 사라졌다'
    _cnt = len(list((OUT/'labels'/'train').glob('*.txt')))
    print(f'labels ← 1클래스본 (train {_cnt:,}개) · 원본 2클래스는 {L2} 에 보존')
    print('  ▶ 2클래스로 되돌리려면: labels → labels_1cls 로 rename 후 labels_2cls → labels')

import numpy as _np
_a = []
for f in list((OUT/'labels'/'train').glob('*.txt'))[:20000]:
    for ln in f.read_text().splitlines():
        v = ln.split()
        if len(v) == 5: _a.append((int(float(v[0])), float(v[3])*float(v[4])))
if _a:
    _c = _np.array([x[0] for x in _a]); _ar = _np.array([x[1] for x in _a])
    print('\n■ 학습 라벨 면적(정규화) — 예측이 GT의 112배였던 이유를 여기서 본다')
    for k in sorted(set(_c.tolist())):
        q = _np.percentile(_ar[_c == k], [50, 90, 99])
        print(f'  cls{k} {int((_c==k).sum()):>7,}개  p50 {q[0]:.5f}  p90 {q[1]:.5f}  '
              f'p99 {q[2]:.5f}  (셀 대비 p99 {q[2]:.1%})')

DRIVE_OUT = Path('/content/drive/MyDrive/kt_out/runs_main'); DRIVE_OUT.mkdir(parents=True, exist_ok=True)
CKPT = DRIVE_OUT/RUN/'weights'; CKPT.mkdir(parents=True, exist_ok=True)
def _sync(tr):        # 매 에폭 Drive 백업 (세션 죽어도 회수)
    try:
        w = Path(tr.last).parent
        for p in w.glob('epoch*.pt'):
            if not (CKPT/p.name).exists(): shutil.copy(p, CKPT/p.name)
        for p in (Path(tr.last), Path(tr.best)):
            if p.exists(): shutil.copy(p, CKPT/p.name)
    except Exception as e: print('동기화 스킵:', e)

# ── resume 준비 ───────────────────────────────────────────────────────────
#  ★이어학습 규칙: ultralytics는 last.pt + 같은 project/name 이 있어야 resume 된다.
#    Drive에 백업된 스냅샷을 로컬로 미러하고, last.pt가 없으면 최신 epoch을 승격시킨다.
import shutil
local_run = Path('/content/runs_main')/RUN
local_last = local_run/'weights'/'last.pt'
drive_ckpts = sorted(CKPT.glob('*.pt'))

if not local_last.exists() and drive_ckpts:
    print(f'Drive 스냅샷 {len(drive_ckpts)}개 발견 → 로컬로 미러')
    (local_run/'weights').mkdir(parents=True, exist_ok=True)
    for p in drive_ckpts:
        t = local_run/'weights'/p.name
        if not t.exists(): shutil.copy(p, t)
    if not local_last.exists():                       # last.pt 없으면 최신 epoch 승격
        eps = []
        for p in (local_run/'weights').glob('epoch*.pt'):
            try: eps.append((int(''.join(filter(str.isdigit, p.stem))), p))
            except ValueError: pass
        if eps:
            k, src = max(eps)
            shutil.copy(src, local_last)
            print(f'  last.pt 없음 → epoch{k}.pt를 last.pt로 승격')

RESUME = local_last.exists()
if RESUME:
    import torch as _t
    try:
        _ck = _t.load(local_last, map_location='cpu', weights_only=False)
        print(f'▶ 이어학습: {local_last}  (완료 에폭 {_ck.get("epoch", "?")}, 목표 {EPOCHS})')
        del _ck
    except Exception as e:
        print('▶ 이어학습:', local_last, '(에폭 정보 읽기 실패:', str(e)[:60], ')')
else:
    print(f'▶ 새로 학습 시작 (사전학습 {BASE})')
    assert not drive_ckpts, ('★Drive에 스냅샷이 있는데 이어학습이 안 된다 — '
                             'RUN 이름이 바뀌었는지 확인. 그냥 두면 기존 결과를 덮어쓴다.')

# ── 학습 ─────────────────────────────────────────────────────────────────
m = YOLO(str(local_last) if RESUME else BASE)
m.add_callback('on_fit_epoch_end', _sync)
if RESUME:
    # resume=True 면 학습 인자는 체크포인트에 저장된 것을 그대로 쓴다(변경 불가)
    m.train(resume=True)
else:
    m.train(data=str(yml), imgsz=IMGSZ, batch=BATCH, epochs=EPOCHS, save_period=1,
            project='/content/runs_main', name=RUN, exist_ok=True,
            amp=True, lr0=0.0005, patience=15, hsv_h=0.0, hsv_s=0.0, hsv_v=0.1,
            # ★증강을 v2 그대로 되돌렸다 — v3(mosaic·회전 OFF)는 recall을 반토막 냈고
            degrees=10, translate=0.2, scale=0.1, copy_paste=0.0,
            fliplr=0.5, flipud=0.5,
            mosaic=1.0, close_mosaic=12,   # 14에폭 − 12 = ep2에 mosaic off (v2와 같은 일정)
            val=True, **({'mask_ratio': 2, 'overlap_mask': False} if TASK == 'seg' else {}))
print('학습 완료 →', CKPT)
